# Phase 4 — Requirements + CV score

Extract tools / degree / years from job text, build frequencies, score a sample CV.

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd

cwd = Path.cwd()
project_root = cwd if (cwd / "data" / "processed").exists() else cwd.parent
sys.path.insert(0, str(project_root / "src"))

from extract_requirements import run_for_role, score_cv

ROLE_SLUG = "data_analyst"
run_for_role(ROLE_SLUG)

reqs = pd.read_parquet(project_root / "data" / "processed" / f"job_requirements_{ROLE_SLUG}.parquet")
freq = pd.read_parquet(project_root / "data" / "processed" / f"requirement_frequencies_{ROLE_SLUG}.parquet")
freq

In [ ]:
profile = json.loads(
    (project_root / "data" / "processed" / "sample_cv_profile.json").read_text(encoding="utf-8")
)
result = score_cv(profile, freq)
print("Profile:", profile)
print("Score:", result["score"])
print("Matched:", result["matched_criteria"])
print("Top gaps:")
for g in result["gaps"][:10]:
    print(f"  {g['criterion']}: {100*g['frequency']:.1f}% of sponsor jobs")

## QA sample

Fill `correct` (Y/N) in `data/processed/req_qa_sample.csv`, then run the next cell for precision.

In [ ]:
qa_path = project_root / "data" / "processed" / "req_qa_sample.csv"
qa = pd.read_csv(qa_path)
labeled = qa[qa["correct"].astype(str).str.upper().isin(["Y", "N"])].copy()
if labeled.empty:
    print("No hand labels yet. Open req_qa_sample.csv and set correct to Y or N.")
else:
    labeled["ok"] = labeled["correct"].str.upper().eq("Y")
    print(labeled.groupby("field")["ok"].mean().rename("precision"))